# Inspect Consolidated Soil Moisture Datasets

Load and visualize the four soil moisture source datasets for January 1980.

Sources (see `catalog/variables.yml` → `soil_moisture`):
- MERRA-2 `GWETTOP` — surface soil wetness, dimensionless 0-1, 0-5 cm layer (plant-available water fraction; *not* volumetric water content)
- NLDAS-2 NOAH `SoilM_0_10cm` — total soil water mass, kg m⁻², 0-10 cm layer
- NLDAS-2 MOSAIC `SoilM_0_10cm` — total soil water mass, kg m⁻², 0-10 cm layer
- NCEP/NCAR R1 `soilw_0_10cm` — already volumetric water content (m³/m³) in 0-10 cm layer at ~1.875°; lon stored in 0-360 convention. The upstream file mislabels its units as `kg/m2` — see explanation cell.

The first plot shows raw values in their native units. The "Normalized comparison" section below converts the kg m⁻² sources to volumetric water content (m³/m³), shifts NCEP/NCAR longitudes to -180-180, and clips all four to the NLDAS CONUS footprint for a like-for-like spatial comparison. The explanation cell discusses why values differ across LSMs/forcings/depths.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "1980-01-15"


## Load all four datasets

In [ ]:
datasets = {
    "MERRA-2 (GWETTOP)": {
        "path": DATASTORE / "merra2" / "merra2_consolidated.nc",
        "var": "GWETTOP",
        "units": "fraction (0-1)",
    },
    "NLDAS-2 MOSAIC (SoilM 0-10cm)": {
        "path": DATASTORE / "nldas_mosaic" / "nldas_mosaic_consolidated.nc",
        "var": "SoilM_0_10cm",
        "units": "kg/m²",
    },
    "NLDAS-2 NOAH (SoilM 0-10cm)": {
        "path": DATASTORE / "nldas_noah" / "nldas_noah_consolidated.nc",
        "var": "SoilM_0_10cm",
        "units": "kg/m²",
    },
    "NCEP/NCAR (soilw 0-10cm)": {
        "path": DATASTORE / "ncep_ncar" / "ncep_ncar_consolidated.nc",
        "var": "soilw_0_10cm",
        "units": "m3/m3",
    },
}

# Open each consolidated file and report basic info
opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run consolidation first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")


## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot January 1980

In [ ]:
import pandas as pd

_target_year = pd.Timestamp(TARGET_TIME).year
_target_month = pd.Timestamp(TARGET_TIME).month
_month_start = pd.Timestamp(year=_target_year, month=_target_month, day=1)
_month_end = _month_start + pd.offsets.MonthEnd(0)
_month_label = f"{_target_year}-{_target_month:02d}"


def _select_month(da, year, month):
    """Pick the first timestamp inside the [year, month] calendar window.

    Sources use different timestamp conventions (mid-month for MERRA-2,
    start-of-month for NLDAS, end-of-month for NCEP/NCAR), so a `nearest`
    selection against a single date can pull from the wrong calendar
    month.  Slicing by month and taking the first hit avoids this.
    """
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthEnd(0)
    sliced = da.sel(time=slice(start, end))
    if sliced.time.size == 0:
        raise ValueError(f"No timestamps in {year}-{month:02d}")
    return sliced.isel(time=0)


n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = _select_month(ds[var], _target_year, _target_month)
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlGnBu", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    # Hide unused subplots
    for idx in range(len(opened), 4):
        axes[idx].set_visible(False)

    fig.suptitle(f"Soil Moisture (raw units) - {_month_label}", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


## Normalized comparison (dimensionless, NLDAS CONUS footprint)

Convert each source to a **dimensionless soil-wetness fraction** and clip to the NLDAS CONUS bounding box for a like-for-like comparison.

The native units differ:

- MERRA-2 `GWETTOP` is already dimensionless (0-1). Note that it is **plant-available water fraction**, not volumetric water content — defined as `(W - W_wilt) / (W_sat - W_wilt)`. We pass it through unchanged but flag the conceptual difference in the explanation cell.
- NLDAS NOAH/MOSAIC `SoilM_0_10cm` is total water mass in kg m⁻² for the 0-10 cm layer. Divide by `(layer_depth_m × ρ_water) = 0.10 × 1000 = 100` to obtain volumetric water content in m³/m³.
- NCEP/NCAR R1 `soilw_0_10cm` is **already** volumetric water content (m³/m³) despite the upstream file labeling its units as `kg/m2` — see `catalog/sources.yml:ncep_ncar.variables` notes. Pass through unchanged. Its longitude is stored in 0-360 convention and is shifted to -180-180 before clipping.

Source timestamp conventions also differ — MERRA-2 stores values at mid-month, NLDAS at start-of-month, NCEP/NCAR at end-of-month — so this cell selects the first timestamp falling inside the target calendar month rather than using `nearest()` against a single date.

NLDAS NOAH and MOSAIC are CONUS-only at 0.125°; using their lat/lon range as the clip footprint gives the same geographic view across all four panels and avoids tropical wet maxima from the global products dominating the colour scale.


In [ ]:
import numpy as np


def _to_volumetric_fraction(ds, info, year, month):
    """Convert a source-specific soil-moisture variable to a dimensionless
    soil wetness fraction at the target calendar month.

    - GWETTOP (MERRA-2): already dimensionless plant-available wetness.
    - NLDAS NOAH/MOSAIC (kg/m^2 in 10 cm layer): divide by 100 -> m^3/m^3 VWC.
      (1 kg/m^2 of water = 1 mm depth; depth/layer_depth = VWC.)
    - NCEP/NCAR (m^3/m^3): pass through unchanged. Upstream files
      mislabel `units: kg/m2` but valid_range/long_name confirm VWC.
    """
    var = info["var"]
    units = info["units"]
    da = _select_month(ds[var], year, month)

    if "fraction" in units or units.strip() in ("1", "dimensionless", "m3/m3"):
        return da

    if "kg" in units:
        return da / 100.0

    raise ValueError(f"Unknown units {units!r} for {info['var']}")


def _wrap_lon_to_minus180_180(da):
    """Shift 0-360 longitudes to -180-180 and re-sort.  No-op if already -180-180."""
    lon_dim = "lon" if "lon" in da.dims else "x"
    lon = da[lon_dim]
    if float(lon.max()) > 180:
        new_lon = ((lon + 180) % 360) - 180
        da = da.assign_coords({lon_dim: new_lon}).sortby(lon_dim)
    return da


def _clip_to_bbox(da, lon_range, lat_range):
    """Clip a 2D DataArray to a lon/lat bounding box, handling descending lats."""
    lon_dim = "lon" if "lon" in da.dims else "x"
    lat_dim = "lat" if "lat" in da.dims else "y"
    lat_vals = da[lat_dim].values
    if lat_vals[0] > lat_vals[-1]:  # descending
        lat_slice = slice(lat_range[1], lat_range[0])
    else:
        lat_slice = slice(lat_range[0], lat_range[1])
    return da.sel({lon_dim: slice(*lon_range), lat_dim: lat_slice})


# Reference dataset providing the clip footprint and colour scale
_REF_LABEL = "NLDAS-2 NOAH (SoilM 0-10cm)"

if opened:
    # 1) Normalize every source to a dimensionless wetness fraction
    norm = {}
    for label, (ds, info) in opened.items():
        da = _to_volumetric_fraction(ds, info, _target_year, _target_month)
        da = _wrap_lon_to_minus180_180(da)
        norm[label] = da

    # 2) Use NLDAS-NOAH's lat/lon range as the clip footprint
    if _REF_LABEL in norm:
        ref = norm[_REF_LABEL]
        lon_min, lon_max = float(ref.lon.min()), float(ref.lon.max())
        lat_min, lat_max = float(ref.lat.min()), float(ref.lat.max())
        print(f"NLDAS-NOAH footprint: lon [{lon_min:.2f}, {lon_max:.2f}]  lat [{lat_min:.2f}, {lat_max:.2f}]")
        clipped = {label: _clip_to_bbox(da, (lon_min, lon_max), (lat_min, lat_max)) for label, da in norm.items()}
    else:
        print(f"WARN: {_REF_LABEL} not loaded; skipping NLDAS-footprint clip")
        clipped = norm

    actual_times = {label: str(da.time.values)[:10] for label, da in clipped.items()}
    for label, da in clipped.items():
        print(f"{label} ({actual_times[label]}): mean = {float(da.mean(skipna=True)):.3f} (NLDAS footprint)")

    # 3) Shared colour scale derived from NLDAS-NOAH (CONUS land mask)
    if _REF_LABEL in clipped:
        ref_vals = clipped[_REF_LABEL].values.ravel()
        ref_vals = ref_vals[~np.isnan(ref_vals)]
        vmin = float(np.percentile(ref_vals, 2))
        vmax = float(np.percentile(ref_vals, 98))
        print(f"Colour scale from {_REF_LABEL}: {vmin:.3f} - {vmax:.3f}")
    else:
        all_vals = np.concatenate([c.values.ravel() for c in clipped.values()])
        all_vals = all_vals[~np.isnan(all_vals)]
        vmin = float(np.percentile(all_vals, 2))
        vmax = float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    flat = axes.flatten()
    for idx, (label, da) in enumerate(clipped.items()):
        ax = flat[idx]
        da.plot(ax=ax, cmap="YlGnBu", vmin=vmin, vmax=vmax)
        ax.set_title(f"{label}\n{actual_times[label]} | dimensionless", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(clipped), 4):
        flat[idx].set_visible(False)

    fig.suptitle(
        f"Soil Moisture - dimensionless wetness, NLDAS CONUS footprint - {_month_label}\n"
        f"colour scale from {_REF_LABEL}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No datasets available yet.")


### Why do the four soil-moisture sources differ?

These products do not measure exactly the same thing, and the four LSMs/forcings produce visibly different spatial patterns even after unit normalisation.

**Variable definitions and depths**

- **MERRA-2 `GWETTOP`** is *plant-available surface soil wetness* in the 0-5 cm layer, defined as `(W - W_wilt) / (W_sat - W_wilt)` — a dimensionless 0-1 fraction relative to the wilting point and saturation, NOT volumetric water content. A value of 0 = wilting point, 1 = saturation; values therefore tend to be *higher* than the corresponding VWC at the same physical wetness, and MERRA-2 panels look systematically "wetter" on the shared colour scale.
- **NLDAS-2 NOAH and MOSAIC `SoilM_0_10cm`** is the *total water mass* in the 0-10 cm soil column, in kg m⁻². Divide by `(layer_depth_m × ρ_water) = 0.10 × 1000 = 100` to obtain volumetric water content in m³/m³.
- **NCEP/NCAR R1 `soilw_0_10cm`** is **already volumetric water content** in m³/m³, even though the upstream file mislabels its units as `kg/m2`. The NetCDF's `long_name` ("volumetric soil moisture 0-10 cm"), `var_desc` ("Volumetric Soil Moisture"), and `valid_range` ([0.0, 1.0]) all confirm this; the unit string is a known NCEP R1 GRIB metadata error. Treating it as kg m⁻² and dividing by 100 produces values ~0.001-0.005 — physically impossible for soil moisture and the symptom you would see on a shared colour scale.

**Why the patterns diverge**

1. **Different LSMs.** MERRA-2 uses the GEOS Catchment LSM (TOPMODEL-style groundwater partitioning); NLDAS-MOSAIC uses Mosaic LSM (vegetation-tile averaging); NLDAS-NOAH uses NOAH (multi-layer with frozen-soil scheme); NCEP/NCAR R1 uses an older 2-layer Pan-and-Mahrt scheme. Two LSMs forced with the same precipitation will still produce different soil moisture distributions.

2. **Different forcing precipitation.** MERRA-2 uses MERRA-2 reanalysis precipitation; NLDAS-2 blends NOAA CPC gauge-corrected precipitation into its model; NCEP/NCAR R1 uses NCEP forecast precipitation only. CONUS-mean precipitation can disagree by 10-30% between these products, and soil moisture inherits that disagreement.

3. **Different surface layer thickness.** MERRA-2's 0-5 cm layer responds faster to precipitation/drying than the 0-10 cm layers in the others, especially in summer dry-down or storm-event recovery.

4. **Different conceptual definition.** GWETTOP's plant-available framing produces values typically in the 0.1-0.9 range; VWC from NLDAS/NCEP typically sits in the 0.05-0.45 range. The shared colour scale here makes that offset visible — MERRA-2 panels look wetter by construction even when the actual physical state is identical.

5. **Resolution differences.** NLDAS at 0.125° resolves orographic, basin-scale, and land-cover features; NCEP/NCAR R1 at ~1.875° smears CONUS into roughly 25 grid cells and any small features (mountain ranges, river valleys, lakes) collapse into single-cell averages.

6. **Timestamp conventions.** MERRA-2 timestamps fall mid-month, NLDAS at start-of-month, NCEP/NCAR at end-of-month. A `nearest()` selection against a single mid-month date can land in the wrong calendar month for NCEP/NCAR; this notebook selects the first timestamp falling inside the target calendar month instead.

**Calibration target implication**

The SOM target normalises each source 0-1 independently per calendar month (monthly time step) or over the full 1982-2010 period (annual time step), so the constant offset between GWETTOP-style and VWC-style values, the resolution differences, and the layer-depth differences all cancel out. The optimiser sees four sources sharing the same 0-1 range and targets *relative* spatial-and-temporal pattern, not absolute wetness. A multi-source per-cell min/max range across these four products is then taken as the calibration error bound per HRU. The raw and normalised plots in this notebook are for sanity-checking the spatial pattern of each input, not for reading absolute magnitudes.


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
